<a href="https://colab.research.google.com/github/SabrinaZ600/AI-four-dimension-project/blob/main/sony_trust.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q transformers accelerate sentencepiece

In [5]:
import json
import pandas as pd
import torch

from tqdm import tqdm

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

In [6]:
from google.colab import files

uploaded = files.upload()

df = pd.read_csv(next(iter(uploaded)))

Saving Sony.csv to Sony.csv


In [7]:
df["标题"]=df["标题"].fillna("")

df["内容"]=df["内容"].fillna("")

df["full_text"]=df["标题"]+"\n"+df["内容"]

df=df.drop_duplicates("full_text")

print(len(df))

100


In [11]:
model_name="Qwen/Qwen2.5-0.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name)

model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:
PROMPT="""
You are a marketing researcher.

Read ONE Chinese social media post.

Is it talking about Sony wireless noise cancelling headphones?

If not,

return

{
"is_relevant":false
}

Otherwise return ONLY JSON.

{
"is_relevant":true,
"trust_score":1-5,
"recommendation":true/false,
"repurchase":true/false
}

Post:
"""

In [13]:
def analyze(text):

    messages=[
        {
            "role":"system",
            "content":"You are a branding researcher."
        },
        {
            "role":"user",
            "content":PROMPT+text
        }
    ]

    prompt=tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs=tokenizer(prompt,
                     return_tensors="pt").to(model.device)

    outputs=model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

    answer=tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer

In [14]:
raw=analyze(df.iloc[0]["full_text"])

print(raw)

```json
{
  "is_relevant": true,
  "trust_score": 5,
  "recommendation": true,
  "repurchase": false
}
```


In [15]:
import re

def parse(text):

    m=re.search(r"\{.*?\}",text,re.S)

    if m:

        return json.loads(m.group())

    return None

In [16]:
results=[]

for text in tqdm(df["full_text"]):

    try:

        r=parse(analyze(text))

    except:

        r=None

    results.append(r)

100%|██████████| 100/100 [02:35<00:00,  1.55s/it]


In [17]:
df["trust_score"]=[
None if r is None else r.get("trust_score")
for r in results
]

In [18]:
trust=df["trust_score"].dropna().mean()

print(trust)

4.434343434343434
